# Ejercicio: Multiplicación de Matrices con Machine Learning

 Conformar un dataset con los resultados de 50.000 multiplicaciones de matrices de 2x2, generadas con números enteros aleatorios entre -20 y 20. Entrenar un modelo de machine learning, probarlo con 10 ejemplos nuevos, comparar los resultados con el método analítico y calcular la diferencia en el costo computacional.

In [1]:
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

## 1. Generación del Dataset
Se generan 50.000 pares de matrices de 2x2 con valores enteros aleatorios entre -20 y 20, inclusive. Cada par es una entrada y su producto calculado con NumPy es la salida correcta (etiqueta), por lo que se plantea como un problema de regresión supervisada.

Para alimentar la red, se concatenan los cuatro elementos de cada matriz de entrada: cada ejemplo queda representado por 8 características. El producto 2x2 se convierte en un vector de 4 valores objetivo. `reshape` conserva el orden por filas de NumPy. Los datos aleatorios no tienen una semilla fija, así que cambian al ejecutar nuevamente esta sección.

In [2]:
# Cantidad de pares de matrices que formarán el dataset.
num_samples = 50000

# Generar los operandos A y B. randint no incluye el límite superior;
# por eso usamos 21 para que el valor 20 también pueda aparecer.
A = np.random.randint(-20, 21, size=(num_samples, 2, 2))
B = np.random.randint(-20, 21, size=(num_samples, 2, 2))

# Calcular el producto exacto de cada par; estos resultados son las
# respuestas objetivo que la red intentará aprender a aproximar.
C = np.matmul(A, B)

# Aplanar cada matriz de entrada por filas y concatenar A con B.
# X tiene una fila por ejemplo y 8 columnas: 4 valores de A + 4 de B.
X = np.concatenate((A.reshape(num_samples, 4), B.reshape(num_samples, 4)), axis=1)

# y contiene los 4 elementos del producto C para cada ejemplo.
y = C.reshape(num_samples, 4)

## 2. Entrenamiento del Modelo
Cada elemento del producto combina multiplicaciones y sumas de las entradas, por ejemplo $C_{11} = A_{11} \cdot B_{11} + A_{12} \cdot B_{21}$. Usaremos un Perceptrón Multicapa (`MLPRegressor`) con dos capas ocultas para aproximar esta relación. Como es un modelo de regresión, puede producir valores decimales y no garantiza resultados enteros exactos. Se evalúa con el error cuadrático medio (MSE), el promedio de los errores al cuadrado para los valores predichos: cuanto menor sea, más cercanas son en promedio las predicciones a las respuestas. La partición reserva el 20 % de los ejemplos para evaluar datos que no participaron en el ajuste.

Si aparece `ConvergenceWarning` al alcanzar `max_iter`, significa que el optimizador terminó el límite de iteraciones sin declarar convergencia. El modelo sí produce predicciones, pero el MSE obtenido corresponde a ese entrenamiento y podría cambiar si se ajusta su configuración.

In [3]:
# Separar ejemplos para que la evaluación use datos que no participaron
# en el ajuste. random_state fija esta partición para reproducir el resultado.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Red con dos capas ocultas de 128 y 64 neuronas. ReLU introduce
# no linealidad; max_iter limita las iteraciones de entrenamiento.
# random_state fija la inicialización aleatoria de la red.
modelo = MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu', max_iter=500, random_state=42)

print("Entrenando el modelo... (esto puede tomar unos segundos)")
# fit ajusta los pesos para aproximar el mapeo desde 8 entradas a 4 salidas.
modelo.fit(X_train, y_train)

# Predecir productos para el conjunto retenido y calcular el promedio
# del cuadrado de los errores de sus componentes.
y_pred = modelo.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Error Cuadrático Medio (MSE) en Test: {mse:.4f}")

Entrenando el modelo... (esto puede tomar unos segundos)
Error Cuadrático Medio (MSE) en Test: 57.1632


c:\Users\tique\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


## 3. Comparación con 10 Ejemplos
Generamos 10 pares de matrices distintos a los del dataset y comparamos dos resultados: el producto exacto calculado por `np.matmul` y la aproximación de la red. En cada comparación, ambos resultados tienen forma 2x2. Esta pequeña muestra ilustra las salidas, pero no reemplaza una evaluación estadística amplia.

In [4]:
# Crear 10 pares nuevos (distintos de los usados para entrenar).
A_test = np.random.randint(-20, 21, size=(10, 2, 2))
B_test = np.random.randint(-20, 21, size=(10, 2, 2))

# Usar la misma representación que en entrenamiento: 4 elementos de A
# seguidos por los 4 elementos de B.
X_10 = np.concatenate((A_test.reshape(10, 4), B_test.reshape(10, 4)), axis=1)

# Obtener la referencia exacta y la estimación del modelo, respectivamente.
analitico = np.matmul(A_test, B_test)
predicciones = modelo.predict(X_10)

# Mostrar cada referencia exacta junto a la estimación. La red entrega
# cuatro valores por ejemplo; reshape permite verlos como matriz 2x2.
for i in range(10):
    print(f"--- Ejemplo {i+1} ---")
    print("Resultado Analítico:\n", analitico[i])
    # Redondeamos la predicción de la red neuronal a 2 decimales para una lectura más limpia
    print("Predicción del Modelo:\n", np.round(predicciones[i].reshape(2, 2), 2))
    print()

--- Ejemplo 1 ---
Resultado Analítico:
 [[ 44 352]
 [250 168]]
Predicción del Modelo:
 [[ 37.61 361.91]
 [250.46 158.6 ]]

--- Ejemplo 2 ---
Resultado Analítico:
 [[108  55]
 [216  90]]
Predicción del Modelo:
 [[110.72  49.68]
 [204.46  94.42]]

--- Ejemplo 3 ---
Resultado Analítico:
 [[-210  -74]
 [-158   50]]
Predicción del Modelo:
 [[-209.81  -71.83]
 [-153.32   48.71]]

--- Ejemplo 4 ---
Resultado Analítico:
 [[126 -22]
 [  6  54]]
Predicción del Modelo:
 [[116.87 -16.04]
 [ 10.93  51.06]]

--- Ejemplo 5 ---
Resultado Analítico:
 [[-191  275]
 [ -19   15]]
Predicción del Modelo:
 [[-197.16  281.59]
 [ -21.42   26.78]]

--- Ejemplo 6 ---
Resultado Analítico:
 [[306 184]
 [-99 184]]
Predicción del Modelo:
 [[ 301.07  193.8 ]
 [-116.46  181.16]]

--- Ejemplo 7 ---
Resultado Analítico:
 [[ 133 -349]
 [ 140 -356]]
Predicción del Modelo:
 [[ 112.79 -356.27]
 [ 134.89 -347.2 ]]

--- Ejemplo 8 ---
Resultado Analítico:
 [[-127   29]
 [-174  202]]
Predicción del Modelo:
 [[-121.52   28.58]
 

## 4. Cálculo del Costo Computacional
Medimos cuánto tardan `np.matmul` y `modelo.predict` en procesar un lote de 100.000 pares. Las matrices y los vectores de entrada se preparan antes de iniciar el cronómetro; por tanto, se compara el tiempo del producto con el de inferencia, no el tiempo de generación de datos ni el entrenamiento. Los tiempos dependen del equipo y de la carga del sistema, y esta comparación por lotes no equivale al costo de una sola operación.

In [5]:
# Preparar el mismo lote de operandos para ambos métodos.
A_bench = np.random.randint(-20, 21, size=(100000, 2, 2))
B_bench = np.random.randint(-20, 21, size=(100000, 2, 2))
X_bench = np.concatenate((A_bench.reshape(100000, 4), B_bench.reshape(100000, 4)), axis=1)

# 1. Cronometrar el producto matricial vectorizado de NumPy.
inicio = time.time()
np.matmul(A_bench, B_bench)
tiempo_analitico = time.time() - inicio

# 2. Cronometrar la inferencia de la red neuronal sobre las mismas entradas.
inicio = time.time()
modelo.predict(X_bench)
tiempo_modelo = time.time() - inicio

print(f"Tiempo operación analítica: {tiempo_analitico:.6f} segundos")
print(f"Tiempo predicción del modelo: {tiempo_modelo:.6f} segundos")
# La razón resume esta ejecución; no es una constante universal y puede
# variar según el equipo y la carga del sistema.
print(f"Diferencia: El modelo es aproximadamente {tiempo_modelo/tiempo_analitico:.1f} veces más costoso computacionalmente.")

Tiempo operación analítica: 0.004158 segundos
Tiempo predicción del modelo: 0.335715 segundos
Diferencia: El modelo es aproximadamente 80.7 veces más costoso computacionalmente.


### Conclusión del Costo Computacional
La multiplicación analítica de matrices 2x2 requiere 8 multiplicaciones y 4 sumas por par; además, NumPy ejecuta estas operaciones de forma directa y vectorizada. La red neuronal realiza cálculos en sus capas para aproximar el resultado y necesita muchas más operaciones por predicción. Para esta tarea conocida, el método analítico es más eficiente y exacto: el modelo es un ejemplo didáctico de aprendizaje, no una alternativa práctica al producto matricial. La razón impresa arriba solo describe el tiempo medido en esta ejecución y puede cambiar según el equipo y la carga.